In [1]:
import pandas as pd
import pulp as plp
solver =  plp.GLPK_CMD(msg=0)

df_parcelle = pd.read_csv('../data/parcelles.csv', index_col=0)
df_marches = pd.read_csv('../data/marches.csv', index_col=0)
df_transport = pd.read_csv('../data/couts_transport.csv', index_col=0)
df_temps = pd.read_csv('../data/temps_transport.csv', index_col=0)
df_parametres = pd.read_csv('../data/parametres.csv', index_col=0)


parcelles = list(df_parcelle.index)
marches  = list(df_marches.index)

profit = {}
production = df_parcelle['production'].to_dict()
demande = df_marches['demande_max'].to_dict()
capacite_transport = df_parametres.loc['capacite_transport', 'valeur']

for i in parcelles:
    for j in marches:
        p_j = df_marches.loc[j, 'prix']
        c_ij = df_transport.loc[i, j]
        t_ij = df_temps.loc[i, j]
        profit[i, j] = p_j*(1 - t_ij*0.02) - c_ij


model = plp.LpProblem('Agricole_Exact', plp.LpMaximize)

# Variables
x_exact = plp.LpVariable.dicts('x', 
    [(i, j) for i in parcelles for j in marches], 
    lowBound=0, cat='Continuous')

# Fonction objectif
model += plp.lpSum(profit[i, j] * x_exact[i, j] 
                   for i in parcelles for j in marches)

# Contraintes
for i in parcelles:
    model += plp.lpSum(x_exact[i, j] for j in marches) <= production[i]

for j in marches:
    model += plp.lpSum(x_exact[i, j] * (1 - 0.02 * df_temps.loc[i, j]) 
                       for i in parcelles) <= demande[j]

model += plp.lpSum(x_exact[i, j] 
                   for i in parcelles for j in marches) <= capacite_transport

# Résolution
model.solve(solver)
profit_exact = plp.value(model.objective)
for i in parcelles:
    for j in marches:
        if plp.value(x_exact[i, j]) > 0:
            reçue = plp.value(x_exact[i, j]) * (1 - 0.02 * df_temps.loc[i, j])
            print(f"  {i} vers {j} : {plp.value(x_exact[i,j]):.2f} t expédiées, {reçue:.2f} t reçues")
print(f"Profit   : {profit_exact:.0f} FCFA")


  P2 vers M1 : 4.80 t expédiées, 4.61 t reçues
  P2 vers M2 : 1.20 t expédiées, 1.18 t reçues
  P3 vers M3 : 6.00 t expédiées, 5.76 t reçues
  P5 vers M1 : 5.62 t expédiées, 5.39 t reçues
  P5 vers M3 : 2.38 t expédiées, 2.24 t reçues
Profit   : 6093445 FCFA
